# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = dataset.metadata.to_json()

print(f"Dataset name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print("Metadata overview (fields):")
pprint.pprint(list(metadata_json.keys()))

## 2. Data Overview
Review available record sets and their fields. All `@id` values are used to reference dataset entities.

In [ ]:
# List the available record sets and their @id
all_record_sets = dataset.metadata.record_sets

print("Record Sets found in metadata:")
for recset in all_record_sets:
    print(f"@id: {recset.id}, name: {recset.name}")

# Optionally, list the fields within each record set
print("\nExample: Fields for each record set (by @id):")
for recset in all_record_sets:
    print(f"\nRecord set @id: {recset.id}, name: {recset.name}")
    if hasattr(recset, 'fields') and recset.fields:
        for field in recset.fields:
            print(f"  Field @id: {field.id}  name: {field.name}  dataType: {field.data_type}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references below use record set and field `@id` values from the overview above.

In [ ]:
# Extract data from each record set and load to DataFrames
record_sets_to_load = [recset.id for recset in dataset.metadata.record_sets]
dataframes = {}

for recset_id in record_sets_to_load:
    records = list(dataset.records(record_set=recset_id))
    if len(records) > 0:
        dataframes[recset_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from record set {recset_id}.")
    else:
        print(f"No records found for record set {recset_id}.")

# Pick first available and non-empty record set for exploration
primary_recset_id = None
for recset_id, df in dataframes.items():
    if not df.empty:
        primary_recset_id = recset_id
        break

if primary_recset_id is not None:
    print(f"\nShowing the fields/columns for record set {primary_recset_id}:")
    print(dataframes[primary_recset_id].columns.tolist())
    display(dataframes[primary_recset_id].head())
else:
    print("No non-empty record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All column and field references use their unique `@id` as shown in the previous section.

In [ ]:
import numpy as np

# Pick a numeric field by examining the columns; here we look for a likely candidate
df = dataframes[primary_recset_id]
numeric_candidates = []
for col in df.columns:
    # Heuristically choose columns that are likely numeric (float or int)
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_candidates.append(col)

print(f"Numeric candidate columns: {numeric_candidates}")
if numeric_candidates:
    numeric_field = numeric_candidates[0]  # Select first numeric field
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}):")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Pick a group field if any categorical columns exist
    categorical_candidates = [col for col in df.columns
                             if df[col].dtype == object and not pd.api.types.is_numeric_dtype(df[col])]
    group_field = None
    if categorical_candidates:
        group_field = categorical_candidates[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable group (categorical) field found for grouping.")
else:
    print("No numeric fields available for analysis in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using their `@id` columns.

In [ ]:
import matplotlib.pyplot as plt

if numeric_candidates:
    plt.figure(figsize=(8, 4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10, 4))
        df.boxplot(column=numeric_field, by=group_field, grid=False, rot=90)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated the use of the `mlcroissant` library for loading and exploring a Mass Spectrometry protein dataset defined via the Croissant schema. We reviewed the dataset structure using `@id` identifiers, loaded data to DataFrames, filtered and normalized numerical fields, and visualized relationships between fields. All field and record references were made via their `@id` per the dataset schema definition.

Further analysis can be performed by referring to additional record sets and fields by their `@id`, and extending the exploratory and visualization steps for more in-depth domain research.